# 02 — WGAN-GP Training for IoT Traffic Synthesis

This notebook trains a WGAN-GP model to generate synthetic IoT traffic windows with shape `(50, 41)`.

In [ ]:
# Cell 1: Imports and project path setup
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader, TensorDataset

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from backend.models.wgan_gp import WGANGP, DEFAULT_CONFIG

In [ ]:
# Cell 2: Load preprocessed training windows
processed_candidates = [
    project_root / 'backend' / 'data' / 'processed',
    project_root / 'data' / 'processed',
]
processed_dir = next((path for path in processed_candidates if path.exists()), None)
if processed_dir is None:
    raise FileNotFoundError('Processed data directory not found.')

X_train = np.load(processed_dir / 'X_train.npy')
print('Processed directory:', processed_dir)
print('X_train shape:', X_train.shape)

In [ ]:
# Cell 3: Build dataloader and initialize WGANGP
batch_size = 64
epochs = 100

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.float32)),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

config = DEFAULT_CONFIG.copy()
config.update({
    'seq_len': int(X_train.shape[1]),
    'n_features': int(X_train.shape[2]),
    'latent_dim': 128,
    'hidden_dim': 256,
    'lambda_gp': 10.0,
    'n_critic': 5,
})

wgan = WGANGP(config=config)
print('Device:', wgan.device)

In [ ]:
# Cell 4: Train WGAN-GP
history = {
    'critic_loss': [],
    'generator_loss': [],
    'gradient_penalty': [],
    'wasserstein': [],
}

for epoch in range(1, epochs + 1):
    running = {k: 0.0 for k in history}
    steps = 0

    for (real_batch,) in train_loader:
        metrics = wgan.train_step(real_batch)
        for key in running:
            running[key] += float(metrics[key])
        steps += 1

    epoch_metrics = {k: v / max(steps, 1) for k, v in running.items()}
    for key, value in epoch_metrics.items():
        history[key].append(value)

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:03d}/{epochs} | "
            f"critic={epoch_metrics['critic_loss']:.6f} | "
            f"gen={epoch_metrics['generator_loss']:.6f} | "
            f"gp={epoch_metrics['gradient_penalty']:.6f}"
        )

In [ ]:
# Cell 5: Plot training curves
plt.figure(figsize=(12, 5))
plt.plot(history['critic_loss'], label='Critic Loss')
plt.plot(history['generator_loss'], label='Generator Loss')
plt.plot(history['gradient_penalty'], label='Gradient Penalty')
plt.title('WGAN-GP Training Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Generate synthetic samples and compare summary stats
wgan.generator.eval()
with torch.no_grad():
    fake_samples = wgan.generate(n_samples=100).cpu().numpy()

real_subset = X_train[:100]

real_mean = float(real_subset.mean())
fake_mean = float(fake_samples.mean())
real_std = float(real_subset.std())
fake_std = float(fake_samples.std())

print(f'Real mean/std: {real_mean:.6f} / {real_std:.6f}')
print(f'Fake mean/std: {fake_mean:.6f} / {fake_std:.6f}')
print('Generated shape:', fake_samples.shape)

In [ ]:
# Cell 7: Save weights
weights_path = project_root / 'backend' / 'models' / 'weights' / 'gan.pt'
wgan.save_weights(weights_path)
print('Saved GAN checkpoint to:', weights_path)